# Prompt Kaynaklari Wiki Sync

Agent-description/agent-mimarisi (single-agent vs. multi-agent) akisi icin
secilen 2 statik referans kaynagini (bkz. AGENTS.md) ceker, ozetler ve
Azure DevOps Wiki'ye (`/Prompt-Kaynaklari/...`) yazar. Bu sayfalar daha
sonra ilgili agent'in system prompt'una statik olarak gomulmek uzere
kullanilir.

Kaynaklar (ikisi de Anthropic Engineering blog, vendor):
1. Building Effective Agents — workflow vs. agent ayrimi, ne zaman
   single-agent/workflow ne zaman multi-agent kullanilmali, agent-computer
   interface (tool/description yazimi) prensipleri.
2. How We Built Our Multi-Agent Research System — orchestrator-worker
   mimarisi, subagent'lara verilen task description'in nasil yazilmasi
   gerektigi, effort/scale kurallari.

robots.txt ve lisans kontrolu onceden yapildi (bkz. AGENTS.md):
- anthropic.com: `robots.txt` -> `User-Agent: *` / `Allow: /` (serbest)


In [0]:
%pip install beautifulsoup4

In [0]:
%run "./Utils"

## Kaynak tanimlari

In [0]:
import json
from datetime import datetime, timezone

from bs4 import BeautifulSoup


SOURCES = [
    {
        "name": "Anthropic - Building Effective Agents",
        "category": "vendor",
        "url": "https://www.anthropic.com/engineering/building-effective-agents",
        "license": "Anthropic PBC, dahili referans amacli. Disariya yeniden yayinlanmamali.",
        "wiki_path": "/Prompt-Kaynaklari/Anthropic-Building-Effective-Agents",
    },
    {
        "name": "Anthropic - How We Built Our Multi-Agent Research System",
        "category": "vendor",
        "url": "https://www.anthropic.com/engineering/multi-agent-research-system",
        "license": "Anthropic PBC, dahili referans amacli. Disariya yeniden yayinlanmamali.",
        "wiki_path": "/Prompt-Kaynaklari/Anthropic-Multi-Agent-Research-System",
    },
]


## robots.txt dogrulamasi (calisma zamaninda)

In [0]:
for source in SOURCES:
    allowed = check_robots_allowed(source["url"])
    print(f"{source['name']}: {'ALLOWED' if allowed else 'DISALLOWED'}")


## Anthropic Engineering blog - icerik cikarma

`anthropic.com/engineering/...` sayfalari, `platform.claude.com`'un
aksine, sunucu tarafinda tam HTML olarak render ediliyor: makale govdesi
`<article>` etiketi icinde okunabilir `<h2>/<h3>/<p>/<li>` etiketleriyle
geliyor. Bu yuzden regex tabanli JS-payload cikarimina gerek yok;
BeautifulSoup ile dogrudan parse edilip markdown'a cevriliyor. Ic ice
gecmis listelerde tekrari onlemek icin bir `<li>`'nin kendi metni,
icindeki alt `<ul>/<ol>` haric hesaplaniyor (alt liste ogeleri ayrica
kendi `<li>` olarak zaten islenecek).


In [0]:
def extract_list_item_text(list_item):

    parts = []

    for child in list_item.children:
        if getattr(child, "name", None) in ("ul", "ol"):
            continue
        if isinstance(child, str):
            parts.append(child)
        else:
            parts.append(child.get_text(" ", strip=True))

    return " ".join(part.strip() for part in parts if part.strip())


def extract_engineering_article_content(raw_html):

    soup = BeautifulSoup(raw_html, "html.parser")
    article = soup.find("article")

    if article is None:
        return ""

    lines = []

    for element in article.find_all(["h2", "h3", "h4", "p", "li"]):

        if element.name == "li":
            text = extract_list_item_text(element)
        else:
            text = element.get_text(" ", strip=True)

        if not text:
            continue

        if element.name == "h2":
            lines.append(f"\n## {text}\n")
        elif element.name == "h3":
            lines.append(f"\n### {text}\n")
        elif element.name == "h4":
            lines.append(f"\n#### {text}\n")
        elif element.name == "li":
            lines.append(f"- {text}")
        else:
            lines.append(text)

    return "\n\n".join(line.strip() for line in lines if line.strip())


## Agent-friendly wiki icerik olusturucu

In [0]:
def build_reference_wiki_content(source_name, source_url, license_text, fetched_at, body_text):

    metadata = {
        "source_name": source_name,
        "source_url": source_url,
        "license": license_text,
        "fetched_at": fetched_at,
        "purpose": "agent-description/agent-mimarisi (single-agent vs multi-agent) akisi icin statik referans kaynagi",
    }

    metadata_block = json.dumps(metadata, ensure_ascii=False, indent=2)

    return (
        f"# {source_name}\n\n"
        f"```json\n{metadata_block}\n```\n\n"
        f"## Icerik\n\n{body_text}\n"
    )


## Wiki'ye yazma

In [0]:
fetched_at = datetime.now(timezone.utc).isoformat()

for source in SOURCES:

    raw_html = fetch_url_text(source["url"])
    body_text = extract_engineering_article_content(raw_html)

    content = build_reference_wiki_content(
        source["name"],
        source["url"],
        source["license"],
        fetched_at,
        body_text,
    )

    push_wiki_page(source["wiki_path"], content)


## Index sayfasi

In [0]:
index_content = (
    "# Prompt Yazma Referans Kaynaklari\n\n"
    "Agent-description/agent-mimarisi (single-agent vs multi-agent) akisi icin "
    "statik olarak gomulen 2 kaynak:\n\n"
    "1. [Anthropic - Building Effective Agents](/Prompt-Kaynaklari/Anthropic-Building-Effective-Agents)\n"
    "2. [Anthropic - How We Built Our Multi-Agent Research System](/Prompt-Kaynaklari/Anthropic-Multi-Agent-Research-System)\n\n"
    f"Son senkronizasyon: {fetched_at}\n"
)

push_wiki_page("/Prompt-Kaynaklari", index_content)
